In [0]:
df_products = spark.table("`e-commerce_sales_catalog`.bronze.bronze_products")

In [0]:
df_products.printSchema()
print("Initial Count:", df_products.count())

In [0]:
df_products = df_products.dropDuplicates(["product_id"])

In [0]:
from pyspark.sql.functions import col

df_products.select([
    col(c).isNull().alias(c) for c in df_products.columns
]).show()

In [0]:
from pyspark.sql.functions import coalesce, lit

df_products = df_products.withColumn(
    "product_category_name",
    coalesce(col("product_category_name"), lit("unknown"))
)

In [0]:
df_products = df_products.fillna({
    "product_name_lenght": "0",
    "product_description_lenght": "0",
    "product_photos_qty": "0",
    "product_weight_g": "0",
    "product_length_cm": "0",
    "product_height_cm": "0",
    "product_width_cm": "0"
})

In [0]:
from pyspark.sql.functions import initcap

df_products = df_products.withColumn(
    "product_category_name",
    initcap(col("product_category_name"))
)

In [0]:
from pyspark.sql.functions import col

df_products = df_products.withColumn("product_weight_g", col("product_weight_g").cast("int")) \
                         .withColumn("product_length_cm", col("product_length_cm").cast("int")) \
                         .withColumn("product_height_cm", col("product_height_cm").cast("int")) \
                         .withColumn("product_width_cm", col("product_width_cm").cast("int"))

In [0]:
df_products.display(5)
print("Final Count:", df_products.count())

In [0]:
from pyspark.sql.functions import col

df_products = df_products.withColumn("product_name_lenght", col("product_name_lenght").cast("int")) \
                         .withColumn("product_description_lenght", col("product_description_lenght").cast("int")) \
                         .withColumn("product_photos_qty", col("product_photos_qty").cast("int"))

In [0]:
df_products = df_products.withColumnRenamed("product_name_lenght", "product_name_length") \
                         .withColumnRenamed("product_description_lenght", "product_description_length")

In [0]:
df_products.display(5)

In [0]:
df_products.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("`e-commerce_sales_catalog`.silver.silver_products")